In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import random
import shutil

DATASET_DIR = r"/content/drive/MyDrive/HDADDON"

TEST_SIZE = 0.5

RANDOM_SEED = 42

def create_test_dataset(dataset_dir, test_size, seed=42):
    random.seed(seed)

    val_img_dir = os.path.join(dataset_dir, "images", "val")
    val_lbl_dir = os.path.join(dataset_dir, "labels", "val")

    test_img_dir = os.path.join(dataset_dir, "images", "test")
    os.makedirs(test_img_dir, exist_ok=True)

    if not os.path.exists(val_img_dir):
        raise FileNotFoundError(f"Source validation directory not found: {val_img_dir}")

    valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
    image_files = [f for f in os.listdir(val_img_dir) if f.lower().endswith(valid_extensions)]

    if not image_files:
        print("No images found in the validation directory.")
        return

    if isinstance(test_size, float) and 0 < test_size < 1:
        num_test = int(len(image_files) * test_size)
    elif isinstance(test_size, int) and test_size > 0:
        num_test = min(test_size, len(image_files))
    else:
        raise ValueError("TEST_SIZE must be a float between 0 and 1 or a positive integer.")

    test_images = random.sample(image_files, num_test)

    moved_images_count = 0
    removed_labels_count = 0

    for img_name in test_images:
        src_img_path = os.path.join(val_img_dir, img_name)
        dst_img_path = os.path.join(test_img_dir, img_name)

        shutil.move(src_img_path, dst_img_path)
        moved_images_count += 1
        base_name = os.path.splitext(img_name)[0]

        for ext in ['.txt', '.xml', '.json']:
            label_name = base_name + ext
            label_path = os.path.join(val_lbl_dir, label_name)

            if os.path.exists(label_path):
                os.remove(label_path)
                removed_labels_count += 1
                break

    print(f"--- Task Complete ---")
    print(f"Successfully moved {moved_images_count} images to: {test_img_dir}")
    print(f"Successfully removed {removed_labels_count} corresponding labels from: {val_lbl_dir}")

if __name__ == "__main__":
    create_test_dataset(DATASET_DIR, TEST_SIZE, RANDOM_SEED)

--- Task Complete ---
Successfully moved 138 images to: /content/drive/MyDrive/HDADDON/images/test
Successfully removed 138 corresponding labels from: /content/drive/MyDrive/HDADDON/labels/val


**SET DATASET PATH**

In [ ]:
dataset_path = "/content/drive/MyDrive/HDADDON"
images_dir = "/content/drive/MyDrive/HDADDON/images"
ann_dir = "/content/drive/MyDrive/HDADDON/annotations"

**VERIFY IMAGES + XML FILES EXIST**

In [ ]:
import os
print("Images : ", len(os.listdir(images_dir)))
print("Annotations : ", len(os.listdir(ann_dir)))

Images :  2
Annotations :  1376


**CONVERT XML → YOLO TXT LABELS, DONT RUN**

In [ ]:
import xml.etree.ElementTree as ET

In [ ]:
yolo_labels = f"{dataset_path}/labels"
os.makedirs(yolo_labels, exist_ok=True)

In [ ]:
classes = {
    "with helmet" : 0,
    "without helmet" : 1,
}

In [ ]:
def convert_to_yolo(size,box) :
  dw = 1/size[0]
  dh = 1/size[1]
  x_center = (box[0]+box[2])/2
  y_center = (box[1]+box[3])/2
  w = box[2]-box[0]
  h = box[3]-box[1]
  return (x_center*dw , y_center*dh, w*dw , h*dh)

In [ ]:
for xml_file in os.listdir(ann_dir):
  if not xml_file.endswith(".xml"):
    continue
  xml_path = os.path.join(ann_dir,xml_file)
  tree = ET.parse(xml_path)
  root = tree.getroot()
  img_name = root.find('filename').text
  img_id = img_name.rsplit('.', 1)[0]

  size = root.find('size')
  w = int(size.find('width').text)
  h = int(size.find('height').text)

  label_path = f"{yolo_labels}/{img_id}.txt"

  with open(label_path, "w") as f:
    for obj in root.findall('object'):
      cls = obj.find('name').text.lower()
      if cls not in classes:
        continue
      cls_id = classes[cls]
      xml_box = obj.find('bndbox')
      xmin = int(xml_box.find('xmin').text)
      xmax = int(xml_box.find('xmax').text)
      ymin = int(xml_box.find('ymin').text)
      ymax = int(xml_box.find('ymax').text)

      yolo_box = convert_to_yolo((w,h),(xmin,ymin,xmax,ymax))
      f.write(f"{cls_id} {' '.join(map(str, yolo_box))}\n")

  print("Conversion Completed!")


Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion Completed!
Conversion

**Splitting Images and Labels into Train/Val**

In [ ]:
import shutil
from sklearn.model_selection import train_test_split

In [ ]:
import os
for folder in ["images/train","images/val", "labels/train", "labels/val"]:
  os.makedirs(f"{dataset_path}/{folder}", exist_ok=True)


In [ ]:
image_files = [f for f in os.listdir(images_dir)
               if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
train_image, val_image = train_test_split(image_files, test_size=0.20, random_state=42)

In [ ]:
import os
import shutil

dataset_path = "/content/drive/MyDrive/HDADDON"
images_dir = os.path.join(dataset_path, "images")
labelspath = os.path.join(dataset_path, "labels")


def move_files(files, split):
    images_moved = 0
    labels_moved = 0
    labels_missing = 0

    img_dest_dir = os.path.join(images_dir, split)
    label_dest_dir = os.path.join(labelspath, split)
    os.makedirs(img_dest_dir, exist_ok=True)
    os.makedirs(label_dest_dir, exist_ok=True)

    print(f"\n--- Starting move for '{split}' split (Total Files to Move: {len(files)}) ---")

    for img_filename in files:
        img_src = os.path.join(images_dir, img_filename)
        img_dest = os.path.join(img_dest_dir, img_filename)

        try:
            shutil.copy(img_src, img_dest)
            images_moved += 1
        except FileNotFoundError:
            print(f"❌ CRITICAL ERROR: Image not found at source: {img_src}")
            continue

        base_name, _ = os.path.splitext(img_filename)
        label_name = base_name + '.txt'

        label_src = os.path.join(labelspath, label_name)
        label_dest = os.path.join(label_dest_dir, label_name)

        if os.path.exists(label_src):
            shutil.copy(label_src, label_dest)
            labels_moved += 1
        else:
            print(f"⚠️ Warning: Label file not found for image: {img_filename}. Expected label: {label_name}")
            labels_missing += 1
    print("--------------------------------------------------")
    print(f"✅ Summary for '{split}' split:")
    print(f"   Images moved to {img_dest_dir}: {images_moved}")
    print(f"   Labels moved to {label_dest_dir}: {labels_moved}")
    print(f"   Labels missing/skipped: {labels_missing}")
    print("--------------------------------------------------")

    if labels_missing > 0:
        print("💡 NOTE: Missing labels means some images will be trained without annotations. This is expected if the original data was noisy.")

move_files(train_image, "train")
move_files(val_image, "val")


--- Starting move for 'train' split (Total Files to Move: 1100) ---
--------------------------------------------------
✅ Summary for 'train' split:
   Images moved to /content/drive/MyDrive/HDADDON/images/train: 1100
   Labels moved to /content/drive/MyDrive/HDADDON/labels/train: 1100
   Labels missing/skipped: 0
--------------------------------------------------

--- Starting move for 'val' split (Total Files to Move: 276) ---
--------------------------------------------------
✅ Summary for 'val' split:
   Images moved to /content/drive/MyDrive/HDADDON/images/val: 276
   Labels moved to /content/drive/MyDrive/HDADDON/labels/val: 276
   Labels missing/skipped: 0
--------------------------------------------------


**Checking Classes Distribution**

In [ ]:
import os

labels_path = "/content/drive/MyDrive/HDADDON/labels/train"

counts = {"with_helmet": 0, "without_helmet": 1}

for file in os.listdir(labels_path):
    if file.endswith(".txt"):
        path = os.path.join(labels_path, file)
        with open(path, "r") as f:
            for line in f.readlines():
                class_id = int(line.split()[0])
                if class_id == 0:
                    counts["with_helmet"] += 1
                elif class_id == 1:
                    counts["without_helmet"] += 1

print(counts)


{'with_helmet': 2292, 'without_helmet': 1955}


**Balancing Classes**

In [ ]:
import os
import cv2
import albumentations as A

train_img_dir = "/content/drive/MyDrive/HDADDON/images/train"
train_lbl_dir = "/content/drive/MyDrive/HDADDON/labels/train"

MINORITY_CLASS_ID = 1  # "without helmet"

transform = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.RandomResizedCrop(size=(640, 640), scale=(0.8, 1.0), p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=12, p=0.5),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, p=0.4),
    ],
    bbox_params=A.BboxParams(
        format='yolo',
        label_fields=['class_labels'],
        min_visibility=0.3
    )
)

label_files = [f for f in os.listdir(train_lbl_dir) if f.endswith(".txt")]
augmented_count = 0

for lbl_file in label_files:
    lbl_path = os.path.join(train_lbl_dir, lbl_file)

    bboxes = []
    class_labels = []
    has_minority = False

    with open(lbl_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            cls_id = int(parts[0])
            coords = [float(x) for x in parts[1:]]

            if cls_id == MINORITY_CLASS_ID:
                has_minority = True

            class_labels.append(cls_id)
            bboxes.append(coords)

    if not has_minority:
        continue

    base_name = os.path.splitext(lbl_file)[0]
    img_name = None
    for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.PNG']:
        if os.path.exists(os.path.join(train_img_dir, base_name + ext)):
            img_name = base_name + ext
            break

    if not img_name:
        continue

    img_path = os.path.join(train_img_dir, img_name)
    image = cv2.imread(img_path)
    if image is None:
        continue
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    try:
        transformed = transform(image=image_rgb, bboxes=bboxes, class_labels=class_labels)

        aug_img = transformed['image']
        aug_bboxes = transformed['bboxes']
        aug_classes = transformed['class_labels']

        if MINORITY_CLASS_ID in aug_classes:
            new_base_name = f"{base_name}_aug1"
            new_img_path = os.path.join(train_img_dir, f"{new_base_name}.jpg")
            new_lbl_path = os.path.join(train_lbl_dir, f"{new_base_name}.txt")

            cv2.imwrite(new_img_path, cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR))

            with open(new_lbl_path, "w") as f:
                for cls_id, bbox in zip(aug_classes, aug_bboxes):
                    bbox_str = " ".join([f"{coord:.6f}" for coord in bbox])
                    f.write(f"{int(cls_id)} {bbox_str}\n")

            augmented_count += 1
    except Exception as e:
        print(f"Skipping {lbl_file} due to transformation error: {e}")

print(f"✅ Created {augmented_count} new augmented image-label pairs containing 'without helmet'.")

/usr/local/lib/python3.13/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


✅ Created 431 new augmented image-label pairs containing 'without helmet'.


**Checking Class Distribution After Balancing**

In [ ]:
import os

labels_path = "/content/drive/MyDrive/HDADDON/labels/train"

counts = {"with_helmet": 0, "without_helmet": 1}

for file in os.listdir(labels_path):
    if file.endswith(".txt"):
        path = os.path.join(labels_path, file)
        with open(path, "r") as f:
            for line in f.readlines():
                class_id = int(line.split()[0])
                if class_id == 0:
                    counts["with_helmet"] += 1
                elif class_id == 1:
                    counts["without_helmet"] += 1

print(counts)


{'with_helmet': 2292, 'without_helmet': 1955}


**Creating Yaml File**

In [ ]:
yaml_text = """
path : {dataset}
train : images/train
val : images/val

names:
  0 : with_helmet
  1 : without_helmet
""".format(dataset = dataset_path)
with open("helmet.yaml","w") as f:
  f.write(yaml_text)
print("helmet.yaml created successfully")

helmet.yaml created successfully


In [2]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 7.7 MB/s eta 0:00:00


In [ ]:
len(os.listdir(labels_path))

1531

**Val Class Distribution Check (No Need to Balance)**

In [ ]:
import os

labels_path = "/content/drive/MyDrive/Deep Learning/HelmetDataset/labels/val"

counts = {"with_helmet": 0, "without_helmet": 1}

for file in os.listdir(labels_path):
    if file.endswith(".txt"):
        path = os.path.join(labels_path, file)
        with open(path, "r") as f:
            for line in f.readlines():
                class_id = int(line.split()[0])
                if class_id == 0:
                    counts["with_helmet"] += 1
                elif class_id == 1:
                    counts["without_helmet"] += 1

print(counts)


{'with_helmet': 631, 'without_helmet': 320}


In [ ]:
import ultralytics
ultralytics.checks()

Ultralytics 8.4.144 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.6/112.6 GB disk)


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")

model.train(
    data="helmet.yaml",
    epochs=40,
    patience = 7,
    imgsz=640,
    batch=16,
    lr0=0.008,
    scale=0.6,
    degrees=10.0,
    perspective=0.0005,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    close_mosaic=12,
    weight_decay=0.0005,
    project="helmet_project",
    name="final_generalizable_run",
)

Ultralytics 8.4.144 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=12, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=helmet.yaml, degrees=10.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.008, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=final_generalizable_run-2, nbs=64, nms=None, opset=None, optimi

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x798d4dedb070>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [ ]:
from ultralytics import YOLO

model_path = "runs/detect/helmet_project/final_generalizable_run/weights/best.pt"
model = YOLO(model_path)
model.export(format="onnx", dynamic=True, simplify=True)

print("Successfully loaded and exported model!")

Ultralytics 8.4.144 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
Model summary (fused): 92 layers, 25,840,918 parameters, 0 gradients, 78.7 GFLOPs

PyTorch: starting from 'runs/detect/helmet_project/final_generalizable_run/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 6, 8400) (49.6 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 12 packages in 326ms
Prepared 4 packages in 1.95s
Installed 4 packages in 379ms
 + colorama==0.4.6
 + onnx==1.22.0
 + onnxruntime==1.29.0
 + onnxslim==0.1.96

requirements: AutoUpdate success ✅ 3.3s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset

In [ ]:
from ultralytics import YOLO

model_path = "runs/detect/helmet_project/final_generalizable_run/weights/best.pt"
model = YOLO(model_path)

results = model.predict(
    source="/content/drive/MyDrive/HDADDON/images/test",
    conf=0.25,
    save=True,
    save_txt=True,
    save_conf=True,
    project="helmet_project",
    name="test_predictions",
)

print(f"Predictions completed! Check output folder: helmet_project/test_predictions")

total_helmets = 0
total_no_helmets = 0

for result in results:
    boxes = result.boxes
    for box in boxes:
        cls_id = int(box.cls[0])
        class_name = model.names[cls_id]

        if class_name == "with_helmet":
            total_helmets += 1
        elif class_name == "without_helmet":
            total_no_helmets += 1

print(f"Test Set Detection Summary:")
print(f"• Total 'with_helmet' detected: {total_helmets}")
print(f"• Total 'without_helmet' detected: {total_no_helmets}")


image 1/138 /content/drive/MyDrive/HDADDON/images/test/BikesHelmets101_png_jpg.rf.ee231f3a6efc939da99b1ed72113a856.jpg: 640x640 1 with_helmet, 36.8ms
image 2/138 /content/drive/MyDrive/HDADDON/images/test/BikesHelmets102_png_jpg.rf.7af7ffff3ca6712592446c1127be69f5.jpg: 640x640 2 with_helmets, 37.0ms
image 3/138 /content/drive/MyDrive/HDADDON/images/test/BikesHelmets103_png_jpg.rf.847b02517b5481bda120d5272f53abfd.jpg: 640x640 3 without_helmets, 37.1ms
image 4/138 /content/drive/MyDrive/HDADDON/images/test/BikesHelmets106_png_jpg.rf.e5db4c45bfc871c7ee64ec9d315b0036.jpg: 640x640 3 with_helmets, 36.9ms
image 5/138 /content/drive/MyDrive/HDADDON/images/test/BikesHelmets110_png_jpg.rf.3a7ad5eeeefbf5090dd1c64126075b5e.jpg: 640x640 7 without_helmets, 36.9ms
image 6/138 /content/drive/MyDrive/HDADDON/images/test/BikesHelmets116_png_jpg.rf.7704e4108982ce561d2463f7644764ce.jpg: 640x640 2 with_helmets, 5 without_helmets, 26.9ms
image 7/138 /content/drive/MyDrive/HDADDON/images/test/BikesHelmets13

**Loading trained model for evaluation**

**Seatbelt Part**

**YOLO FORMAT CONVERSION 1**

In [ ]:
import os
import shutil
dataset_path = r"/content/drive/MyDrive/Deep Learning/Seatbelt Dataset"

splits = ["train", "valid", "test"]

images_folder = os.path.join(dataset_path, "images")
annotations_folder = os.path.join(dataset_path, "annotations")

os.makedirs(images_folder, exist_ok=True)
os.makedirs(annotations_folder, exist_ok=True)

for split in splits:
    split_path = os.path.join(dataset_path, split)
    if not os.path.exists(split_path):
        print(f"Skipping missing folder: {split_path}")
        continue

    for file in os.listdir(split_path):
        file_path = os.path.join(split_path, file)
        if file.lower().endswith(".jpg") or file.lower().endswith(".png"):
            dest = os.path.join(images_folder, file)
            shutil.copy(file_path, dest)

        elif file.lower().endswith(".xml"):
            dest = os.path.join(annotations_folder, file)
            shutil.copy(file_path, dest)

    print(f"✔ Completed copying from: {split}")

print("\n✔ All JPG copied to 'images/'")
print("✔ All XML copied to 'annotations/'")
print("Original dataset folders remain unchanged.")


✔ Completed copying from: train
✔ Completed copying from: valid
✔ Completed copying from: test

✔ All JPG copied to 'images/'
✔ All XML copied to 'annotations/'
Original dataset folders remain unchanged.


**YOLO FORMAT CONVERSION 2**

In [ ]:
import os
import xml.etree.ElementTree as ET

ann_dir = r"/content/drive/MyDrive/Deep Learning/Seatbelt Dataset/annotations"

yolo_labels = os.path.join(os.path.dirname(ann_dir), "labels")
os.makedirs(yolo_labels, exist_ok=True)

classes = {
    "person-seatbelt": 0,
    "person-noseatbelt": 1
}

def convert_to_yolo(size, box):
    dw = 1 / size[0]
    dh = 1 / size[1]

    x_center = (box[0] + box[2]) / 2
    y_center = (box[1] + box[3]) / 2
    w = box[2] - box[0]
    h = box[3] - box[1]

    return (x_center * dw, y_center * dh, w * dw, h * dh)

for xml_file in os.listdir(ann_dir):
    if not xml_file.endswith(".xml"):
        continue

    xml_path = os.path.join(ann_dir, xml_file)
    tree = ET.parse(xml_path)
    root = tree.getroot()

    img_name = root.find('filename').text
    img_id = img_name.rsplit('.', 1)[0]

    size = root.find('size')
    w = int(size.find('width').text)
    h = int(size.find('height').text)

    label_path = os.path.join(yolo_labels, f"{img_id}.txt")

    with open(label_path, "w") as f:
        for obj in root.findall('object'):
            cls = obj.find('name').text.lower()

            if cls not in classes:
                continue

            cls_id = classes[cls]

            xml_box = obj.find('bndbox')
            xmin = int(xml_box.find('xmin').text)
            xmax = int(xml_box.find('xmax').text)
            ymin = int(xml_box.find('ymin').text)
            ymax = int(xml_box.find('ymax').text)

            yolo_box = convert_to_yolo((w, h), (xmin, ymin, xmax, ymax))
            f.write(f"{cls_id} {' '.join(map(str, yolo_box))}\n")

    print(f"Converted: {xml_file}")

print("\n✔ All XML files converted to YOLO format (.txt)")
print(f"YOLO labels saved in: {yolo_labels}")


**SPLITTING IMAGES AND LABELS INTO TRAIN AND VAL SETS**

In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split

dataset_path = "/content/drive/MyDrive/Deep Learning/Seatbelt Dataset"
images_dir = f"{dataset_path}/images"
yolo_labels = f"{dataset_path}/labels"

for folder in ["images/train","images/val", "labels/train", "labels/val"]:
    os.makedirs(f"{dataset_path}/{folder}", exist_ok=True)

image_files = [f for f in os.listdir(images_dir) if f.lower().endswith(".jpg")]

train_image, val_image = train_test_split(image_files, test_size=0.20, random_state=42)


def move_files(files, split):

    for img in files:
        img_src = os.path.join(images_dir, img)
        img_dest = os.path.join(images_dir, split, img)

        shutil.copy(img_src, img_dest)
        base = os.path.splitext(img)[0]
        label_name = base + ".txt"

        label_src = os.path.join(yolo_labels, label_name)
        label_dest = os.path.join(yolo_labels, split, label_name)

        shutil.copy(label_src, label_dest)

move_files(train_image,"train")
move_files(val_image,"val")

print("✅ Done splitting and copying!")


✅ Done splitting and copying!


**Testing if all folders got correct number of images/labels**

In [ ]:
import os

dataset_path = "/content/drive/MyDrive/Deep Learning/Seatbelt Dataset"
paths = {
    "Train": {
        "images": os.path.join(dataset_path, "images/train"),
        "labels": os.path.join(dataset_path, "labels/train")
    },
    "Validation": {
        "images": os.path.join(dataset_path, "images/val"),
        "labels": os.path.join(dataset_path, "labels/val")
    }
}

def count_files(folder, extensions):
    return len([
        f for f in os.listdir(folder)
        if os.path.isfile(os.path.join(folder, f)) and f.lower().endswith(extensions)
    ])

for split, paths_dict in paths.items():
    img_count = count_files(paths_dict["images"], (".jpg", ".jpeg", ".png"))
    label_count = count_files(paths_dict["labels"], (".txt",))

    print(f"{split} Set:")
    print(f"  Images: {img_count}")
    print(f"  Labels: {label_count}")
    print("-" * 40)


Train Set:
  Images: 4232
  Labels: 4232
----------------------------------------
Validation Set:
  Images: 1059
  Labels: 1059
----------------------------------------


**CHECKING CLASS DISTRIBUTION**

In [ ]:
import os

class_names = {0: "person-seatbelt", 1: "person-noseatbelt"}

dataset_path = "/content/drive/MyDrive/Deep Learning/Seatbelt Dataset"
paths = {
    "Train": {
        "images": os.path.join(dataset_path, "images/train"),
        "labels": os.path.join(dataset_path, "labels/train")
    },
    "Validation": {
        "images": os.path.join(dataset_path, "images/val"),
        "labels": os.path.join(dataset_path, "labels/val")
    }
}

def count_classes(labels_folder):
    class_counts = {name: 0 for name in class_names.values()}
    total_labels = 0

    for label_file in os.listdir(labels_folder):
        if label_file.lower().endswith(".txt"):
            with open(os.path.join(labels_folder, label_file), "r") as f:
                lines = f.readlines()
                total_labels += len(lines)
                for line in lines:
                    cls_id = int(line.split()[0])
                    class_name = class_names.get(cls_id, f"Unknown({cls_id})")
                    class_counts[class_name] += 1
    return total_labels, class_counts

for split, p in paths.items():
    total_labels, class_counts = count_classes(p["labels"])
    img_count = len([f for f in os.listdir(p["images"]) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
    print(f"{split} Set:")
    print(f"  Images: {img_count}")
    print(f"  Labels: {total_labels}")
    print(f"  Per Class: {class_counts}")
    print("-" * 40)


Train Set:
  Images: 4232
  Labels: 5628
  Per Class: {'person-seatbelt': 3073, 'person-noseatbelt': 2555}
----------------------------------------
Validation Set:
  Images: 1059
  Labels: 1397
  Per Class: {'person-seatbelt': 719, 'person-noseatbelt': 678}
----------------------------------------


**BALANCING CLASSES**

In [ ]:
import os
import random
import shutil
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

dataset_path = "/content/drive/MyDrive/Deep Learning/Seatbelt Dataset"
train_images_path = os.path.join(dataset_path, "images/train")
train_labels_path = os.path.join(dataset_path, "labels/train")

class_names = {0: "person-seatbelt", 1: "person-noseatbelt"}

datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

class_files = {name: [] for name in class_names.values()}

for label_file in os.listdir(train_labels_path):
    if label_file.endswith(".txt"):
        with open(os.path.join(train_labels_path, label_file), "r") as f:
            lines = f.readlines()
        if not lines:
            continue
        ids_in_file = set(int(line.split()[0]) for line in lines)
        for cls_id, cls_name in class_names.items():
            if cls_id in ids_in_file:
                class_files[cls_name].append(label_file)

max_count = max(len(class_files["person-seatbelt"]), len(class_files["person-noseatbelt"]))

for cls_name in class_files:
    current_count = len(class_files[cls_name])
    if current_count < max_count:
        diff = max_count - current_count
        print(f"Augmenting {cls_name} by {diff} images...")
        for i in range(diff):
            orig_label = random.choice(class_files[cls_name])
            for ext in [".png", ".jpg", ".jpeg"]:
                orig_image = orig_label.replace(".txt", ext)
                if os.path.exists(os.path.join(train_images_path, orig_image)):
                    break

            img = tf.keras.preprocessing.image.load_img(os.path.join(train_images_path, orig_image))
            x = tf.keras.preprocessing.image.img_to_array(img)
            x = x.reshape((1,) + x.shape)

            aug_iter = datagen.flow(x, batch_size=1)
            aug_image = next(aug_iter)[0].astype('uint8')

            new_image_name = f"{orig_label.replace('.txt','')}_aug{i}{ext}"
            new_label_name = f"{orig_label.replace('.txt','')}_aug{i}.txt"

            tf.keras.preprocessing.image.save_img(os.path.join(train_images_path, new_image_name), aug_image)
            shutil.copy(os.path.join(train_labels_path, orig_label),
                        os.path.join(train_labels_path, new_label_name))


Augmenting person-noseatbelt by 448 images...


**CHECKING CLASS DISTRIBUTION AFTER BALANCING**

In [ ]:
import os

class_names = {0: "person-seatbelt", 1: "person-noseatbelt"}

dataset_path = "/content/drive/MyDrive/Deep Learning/Seatbelt Dataset"
paths = {
    "Train": {
        "images": os.path.join(dataset_path, "images/train"),
        "labels": os.path.join(dataset_path, "labels/train")
    },
    "Validation": {
        "images": os.path.join(dataset_path, "images/val"),
        "labels": os.path.join(dataset_path, "labels/val")
    }
}

def count_classes(labels_folder):
    class_counts = {name: 0 for name in class_names.values()}
    total_labels = 0

    for label_file in os.listdir(labels_folder):
        if label_file.lower().endswith(".txt"):
            with open(os.path.join(labels_folder, label_file), "r") as f:
                lines = f.readlines()
                total_labels += len(lines)
                for line in lines:
                    cls_id = int(line.split()[0])
                    class_name = class_names.get(cls_id, f"Unknown({cls_id})")
                    class_counts[class_name] += 1
    return total_labels, class_counts

for split, p in paths.items():
    total_labels, class_counts = count_classes(p["labels"])
    img_count = len([f for f in os.listdir(p["images"]) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
    print(f"{split} Set:")
    print(f"  Images: {img_count}")
    print(f"  Labels: {total_labels}")
    print(f"  Per Class: {class_counts}")
    print("-" * 40)


Train Set:
  Images: 4680
  Labels: 6272
  Per Class: {'person-seatbelt': 3191, 'person-noseatbelt': 3081}
----------------------------------------
Validation Set:
  Images: 1059
  Labels: 1397
  Per Class: {'person-seatbelt': 719, 'person-noseatbelt': 678}
----------------------------------------


**CREATING YAML FILE**

In [ ]:
dataset_path = "/content/drive/MyDrive/Deep Learning/Seatbelt Dataset"

yaml_text = """
path : {dataset}
train : images/train
val : images/val

names:
  0 : person-seatbelt
  1 : person-noseatbelt
""".format(dataset=dataset_path)

with open("seatbelt.yaml", "w") as f:
    f.write(yaml_text)

print("seatbelt.yaml created successfully")


seatbelt.yaml created successfully


**TRAINING**

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

model.train(
    data="seatbelt.yaml",
    epochs=15,
    imgsz=640,
    lr0=0.001,
    hsv_h=0.015,
    hsv_s=0.3,
    hsv_v=0.2,
    fliplr=0.2,
    mosaic=0.1,
    mixup=0.0
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.148 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=seatbelt.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.2, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ad03c20b350>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [ ]:
from ultralytics import YOLO

model_path = "/content/runs/detect/train/weights/best.pt"
model = YOLO(model_path)
model.export(format="onnx", dynamic=True, simplify=True)

print("Successfully loaded and exported model!")

In [ ]:
model_path_sb = '/content/drive/MyDrive/Deep Learning/seatbelt_best_model.pt'
modelseatbelt = YOLO(model_path_sb)